In [2]:
import re
import random
import pandas as pd
from langdetect import detect
from tqdm import tqdm
from sklearn.utils import shuffle


# Data Loading

In [3]:
# Load parallel subtitle data (English and French)
with open("OpenSubtitles.en-fr.en", encoding="utf-8") as f_en, \
     open("OpenSubtitles.en-fr.fr", encoding="utf-8") as f_fr:
    en_lines = f_en.readlines()
    fr_lines = f_fr.readlines()

# Randomly sample 10,000 sentence pairs for preprocessing
idx = random.sample(range(len(en_lines)), 10000)
df = pd.DataFrame({
    "src": [en_lines[i].strip() for i in idx],
    "tgt": [fr_lines[i].strip() for i in idx]
})

print(df.head())
print("Total number of samples:", len(df))


                                                 src  \
0                                  Madame President?   
1  Best thing that ever happened to me,splitting ...   
2  He's not that strong, or else you'd be dead by...   
3                           You're the cutest puppy.   
4                              He knows how we work.   

                                                 tgt  
0                             Madame la Présidente ?  
1  La meilleure chose qui me soit arrivée est d'a...  
2          Pas tant que ça. Ou tu ne serais plus là.  
3                   Tu es le plus mignon des chiots.  
4                      Il sait comment on travaille.  
Total number of samples: 10000


# Preprocessing

In [4]:
def clean_text(
    text: str,
    remove_inside_parentheses: bool = True,
    to_lower: bool = True,
) -> str:
    """Clean and normalize subtitle text."""
    if not isinstance(text, str):
        return ""

    # 1. Remove HTML tags and entities
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"&[a-z]+;", "", text)

    # 2. Remove content inside parentheses (optional)
    if remove_inside_parentheses:
        text = re.sub(r"\(.*?\)", "", text)
        text = re.sub(r"\[.*?\]", "", text)
        text = re.sub(r"\{.*?\}", "", text)
    else:
        text = re.sub(r"[\(\)\[\]\{\}]", "", text)

    # 3. Remove timestamps and subtitle-specific notations
    text = re.sub(r"\d{2}:\d{2}:\d{2},\d{3}", "", text)
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"\(.*?\)", "", text)

    # 4. Trim non-word edges and normalize spaces
    text = re.sub(r"^\W+|\W+$", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    # 5. Lowercase (optional)
    if to_lower:
        text = text.lower()

    return text


def is_valid_sentence(text: str) -> bool:
    """Check if a sentence is valid and not noisy."""
    if not text or len(text.split()) < 2:
        return False
    if re.search(r"\d{2}:\d{2}:\d{2}", text):
        return False
    if re.match(r"^\[.*\]$", text):
        return False
    if re.match(r"^\d+$", text):
        return False
    if len(text) > 300:
        return False
    return True


def detect_language_safe(text, default="unk"):
    """Safely detect language (ignore exceptions)."""
    try:
        return detect(text)
    except:
        return default


In [5]:
def preprocess_opensubs(df: pd.DataFrame, src_lang="en", tgt_lang="fr") -> pd.DataFrame:
    """Full preprocessing pipeline for OpenSubtitles dataset."""
    print("Step 1: Cleaning text ...")
    tqdm.pandas()
    df["src"] = df["src"].progress_apply(clean_text)
    df["tgt"] = df["tgt"].progress_apply(clean_text)

    print("Step 2: Removing invalid or noisy sentences ...")
    df = df[df["src"].apply(is_valid_sentence)]
    df = df[df["tgt"].apply(is_valid_sentence)]

    print("Step 3: Language detection (this may take some time) ...")
    df["src_lang"] = df["src"].progress_apply(detect_language_safe)
    df["tgt_lang"] = df["tgt"].progress_apply(detect_language_safe)
    df = df[(df["src_lang"] == src_lang) & (df["tgt_lang"] == tgt_lang)]

    print("Step 4: Filtering by length ...")
    df = df[df["src"].apply(lambda x: 3 <= len(x.split()) <= 80)]
    df = df[df["tgt"].apply(lambda x: 3 <= len(x.split()) <= 80)]

    print("Step 5: Removing duplicates ...")
    df = df.drop_duplicates(subset=["src", "tgt"])

    print("Step 6: Shuffling data ...")
    df = shuffle(df, random_state=42).reset_index(drop=True)

    print(f"Done. Remaining sentence pairs: {len(df):,}")
    return df


In [6]:
clean_df = preprocess_opensubs(df, src_lang="en", tgt_lang="fr")


Step 1: Cleaning text ...


  0%|          | 0/10000 [00:00<?, ?it/s]

100%|██████████| 10000/10000 [00:00<00:00, 180504.98it/s]


Step 2: Removing invalid or noisy sentences ...
Step 3: Language detection (this may take some time) ...


100%|██████████| 8765/8765 [00:19<00:00, 450.67it/s]

Step 4: Filtering by length ...
Step 5: Removing duplicates ...
Step 6: Shuffling data ...
Done. Remaining sentence pairs: 5,804


In [7]:
output_path = "OpenSubtitles_en-fr_clean.csv"
clean_df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to {output_path}")


Cleaned dataset saved to OpenSubtitles_en-fr_clean.csv


In [ ]:
import pandas as pd
from datasets import load_dataset, get_dataset_config_names


def load_opensubs_clean(path="OpenSubtitles_en-fr_clean.csv"):
    df = pd.read_csv(path)
    if "src_lang" not in df.columns: df["src_lang"] = "en"
    if "tgt_lang" not in df.columns: df["tgt_lang"] = "fr"
    df["dataset"] = "opensubs"
    return df[["src", "tgt", "src_lang", "tgt_lang", "dataset"]].copy()

def load_iwslt2017_enfr(split="test"):
    ds_name = "IWSLT/iwslt2017"

    configs = get_dataset_config_names(ds_name)

    enfr_candidates = [c for c in configs if ("en" in c and "fr" in c)]
    config = enfr_candidates[0] if len(enfr_candidates) > 0 else configs[0]

    dset = load_dataset(ds_name, config, split=split)

    df = dset.to_pandas()

    if "translation" in df.columns:
        df["src"] = df["translation"].apply(lambda x: x.get("en", ""))
        df["tgt"] = df["translation"].apply(lambda x: x.get("fr", ""))
    else:
        possible_en = [c for c in df.columns if c.lower() in ["en", "english", "src"]]
        possible_fr = [c for c in df.columns if c.lower() in ["fr", "french", "tgt"]]
        if not possible_en or not possible_fr:
            raise ValueError(f"Could not find translation fields in columns: {df.columns.tolist()}")
        df["src"] = df[possible_en[0]]
        df["tgt"] = df[possible_fr[0]]

    df = df[["src", "tgt"]].copy()
    df["src_lang"] = "en"
    df["tgt_lang"] = "fr"
    df["dataset"] = f"iwslt2017_{split}"

    return df

def clean_parallel_df(df):
    df = df.copy()
    df["src"] = df["src"].apply(clean_text)
    df["tgt"] = df["tgt"].apply(clean_text)

    df = df[df["src"].apply(is_valid_sentence)]
    df = df[df["tgt"].apply(is_valid_sentence)]

    df = df[df["src"].apply(lambda x: 3 <= len(x.split()) <= 80)]
    df = df[df["tgt"].apply(lambda x: 3 <= len(x.split()) <= 80)]

    df = df.drop_duplicates(subset=["src", "tgt"]).reset_index(drop=True)
    return df

opensubs = load_opensubs_clean("OpenSubtitles_en-fr_clean.csv")

iwslt_test = load_iwslt2017_enfr(split="test")
iwslt_dev  = load_iwslt2017_enfr(split="validation")

iwslt_test = clean_parallel_df(iwslt_test)
iwslt_dev  = clean_parallel_df(iwslt_dev)

combined = pd.concat([opensubs, iwslt_test, iwslt_dev], ignore_index=True)

combined_path = "OpenSubtitles+IWSLT_en-fr_clean.csv"
combined.to_csv(combined_path, index=False, encoding="utf-8")
print(f"Saved combined dataset: {combined_path}")
print(combined["dataset"].value_counts())


Saved combined dataset: OpenSubtitles+IWSLT_en-fr_clean.csv
dataset
iwslt2017_test          8236
opensubs                5804
iwslt2017_validation     835
Name: count, dtype: int64
